# 🚀 Image Classification Training - Google Colab Version

**Optimized for Google Colab with Free GPU!**

Features:
- ✅ Mixed Precision Training (2-3x faster)
- ✅ OneCycleLR for better convergence
- ✅ Mixup augmentation for accuracy
- ✅ Google Drive integration
- ✅ Automatic checkpoint saving

---

## 📦 Step 1: Setup Environment

**Run this cell first!** It will:
1. Mount Google Drive
2. Clone your GitHub repository
3. Install dependencies
4. Check GPU availability

In [ ]:
# ============================================================================
# CONFIGURATION - EDIT THESE VALUES
# ============================================================================

# Your GitHub repository URL
GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"

# Repository name (folder name after cloning)
REPO_NAME = "IMAGE_CLASSIFICATION"

# Google Drive paths (if using Drive for dataset)
DRIVE_DATASET_PATH = "/content/drive/My Drive/IMAGE_CLASSIFICATION_DATA/split_dataset"
DRIVE_OUTPUT_PATH = "/content/drive/My Drive/IMAGE_CLASSIFICATION_OUTPUTS"

# ============================================================================

import os
import sys

print("🚀 Starting setup...\n")

# 1. Mount Google Drive
print("📁 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted!\n")

# 2. Clone repository if not exists
if not os.path.exists(f'/content/{REPO_NAME}'):
    print(f"📥 Cloning repository from GitHub...")
    !git clone {GITHUB_REPO}
    print("✅ Repository cloned!\n")
else:
    print(f"✅ Repository already cloned\n")

# 3. Change to repository directory
os.chdir(f'/content/{REPO_NAME}')
print(f"📂 Working directory: {os.getcwd()}\n")

# 4. Install dependencies
print("📦 Installing dependencies...")
!pip install -q timm tqdm scikit-learn seaborn
print("✅ Dependencies installed!\n")

# 5. Check GPU
import torch
print("="*60)
print("🖥️  SYSTEM INFO")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Training will be slow.")
    print("Go to Runtime > Change runtime type > Select GPU")
print("="*60)

print("\n✅ Setup complete!\n")

## 📊 Step 2: Setup Dataset

**Choose ONE option below:**

### Option A: Use dataset from Google Drive (Recommended)
Upload your `split_dataset` folder to Google Drive first, then run this cell.

In [ ]:
# Option A: Link dataset from Google Drive

# Check if dataset exists in Drive
if os.path.exists(DRIVE_DATASET_PATH):
    # Create symbolic link (fast, no copying)
    local_data_link = '/content/split_dataset'
    
    if not os.path.exists(local_data_link):
        !ln -s "{DRIVE_DATASET_PATH}" "{local_data_link}"
        print(f"✅ Dataset linked from Google Drive")
    else:
        print(f"✅ Dataset already linked")
    
    print(f"\n📁 Dataset location: {local_data_link}")
    
    # Verify dataset structure
    !ls -la {local_data_link}
    
    DATA_ROOT = local_data_link
    
else:
    print(f"❌ Dataset not found at: {DRIVE_DATASET_PATH}")
    print(f"\n📤 Please upload your dataset to Google Drive:")
    print(f"   1. Go to https://drive.google.com")
    print(f"   2. Create folder: IMAGE_CLASSIFICATION_DATA")
    print(f"   3. Upload your 'split_dataset' folder there")
    print(f"   Expected structure:")
    print(f"   {DRIVE_DATASET_PATH}/")
    print(f"   ├── train/")
    print(f"   ├── val/")
    print(f"   └── test/")

### Option B: Upload dataset directly to Colab (Alternative)

**Only use if dataset is small (<500MB)**

In [ ]:
# Option B: Upload dataset to Colab (uncomment if using this method)

# from google.colab import files
# import zipfile

# print("📤 Upload your dataset.zip file...")
# uploaded = files.upload()

# # Extract
# for filename in uploaded.keys():
#     print(f"📦 Extracting {filename}...")
#     with zipfile.ZipFile(filename, 'r') as zip_ref:
#         zip_ref.extractall('/content')

# DATA_ROOT = '/content/split_dataset'
# print(f"✅ Dataset extracted to: {DATA_ROOT}")

## ⚙️ Step 3: Configure Training

Adjust settings based on your needs.

In [ ]:
# Import configuration
from config import DATA_CONFIG, MODEL_CONFIG, TRAIN_CONFIG, OUTPUT_CONFIG

# Merge configs
config = {
    **DATA_CONFIG,
    **MODEL_CONFIG,
    **TRAIN_CONFIG,
    **OUTPUT_CONFIG
}

# ============================================================================
# CUSTOMIZE SETTINGS HERE
# ============================================================================

# Dataset
config['data_root'] = DATA_ROOT  # Use the dataset we set up above

# Output (save to Drive for persistence)
os.makedirs(DRIVE_OUTPUT_PATH, exist_ok=True)
config['output_dir'] = DRIVE_OUTPUT_PATH

# Training
config['epochs'] = 30
config['batch_size'] = 64  # Reduce if OOM (try 32 or 16)

# Colab-specific optimizations
config['num_workers'] = 2  # Colab works best with 2
config['use_amp'] = True   # Mixed precision (keep enabled!)
config['use_mixup'] = True  # Better accuracy

# Model (choose one)
config['model_name'] = 'vit_tiny_patch16_224'  # Fast, good accuracy
# config['model_name'] = 'vit_small_patch16_224'  # Better accuracy, slower
# config['model_name'] = 'vit_base_patch16_224'   # Best accuracy, requires more memory

# ============================================================================

# Print configuration
print("\n" + "="*60)
print("📋 TRAINING CONFIGURATION")
print("="*60)
for key, value in config.items():
    print(f"{key:25s}: {value}")
print("="*60 + "\n")

## 📊 Step 4: Load Dataset

Load and visualize the dataset.

In [ ]:
import json
from pathlib import Path
from dataset_loader import create_dataloaders

# Create dataloaders
train_loader, val_loader, test_loader, num_classes, class_names = create_dataloaders(
    config['data_root'],
    config['batch_size'],
    config['num_workers'],
    config['image_size'],
    config.get('crop_size', None),
    config.get('pin_memory', True),
    config.get('persistent_workers', True),
    config.get('prefetch_factor', 2)
)

# Save class names
output_dir = Path(config['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

print(f"\n✅ Dataset loaded successfully!")
print(f"📁 Classes ({num_classes}): {class_names}")

### 🔍 Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
import torch

def denormalize(tensor):
    """Denormalize image for visualization"""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# Get a batch
images, labels = next(iter(train_loader))

# Plot 8 images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    if idx < len(images):
        img = denormalize(images[idx]).permute(1, 2, 0).numpy().clip(0, 1)
        ax.imshow(img)
        ax.set_title(f"Class: {class_names[labels[idx]]}", fontsize=12)
        ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

## 🤖 Step 5: Create Model

In [ ]:
from model import create_model
from train import Trainer, print_training_info

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Create model
model = create_model(
    num_classes,
    config['model_name'],
    config['pretrained'],
    config['freeze_backbone'],
    config.get('dropout', 0.1)
)

# Print detailed info
print_training_info(config, device, train_loader, val_loader, num_classes, model)

## 🏋️ Step 6: Train Model

**This will take 20-60 minutes depending on your dataset and GPU.**

Progress will be shown in real-time!

In [ ]:
# Create trainer
trainer = Trainer(model, train_loader, val_loader, device, config)

# Start training
print("\n" + "="*80)
print(" " * 30 + "🚀 STARTING TRAINING")
print("="*80 + "\n")

trainer.train()

print("\n" + "="*80)
print(" " * 30 + "✅ TRAINING COMPLETE")
print("="*80)
print(f"\n💾 Results saved to: {config['output_dir']}")
print(f"📊 Check Google Drive for outputs!\n")

## 📈 Step 7: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

epochs = range(1, len(trainer.history['train_acc']) + 1)

# Accuracy
axes[0, 0].plot(epochs, [acc * 100 for acc in trainer.history['train_acc']], 'b-', label='Train', linewidth=2)
axes[0, 0].plot(epochs, [acc * 100 for acc in trainer.history['val_acc']], 'r-', label='Validation', linewidth=2)
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Accuracy (%)')
axes[0, 0].set_title('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(epochs, trainer.history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0, 1].plot(epochs, trainer.history['val_loss'], 'r-', label='Validation', linewidth=2)
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Metrics
axes[1, 0].plot(epochs, trainer.history['precision'], 'g-', label='Precision', linewidth=2)
axes[1, 0].plot(epochs, trainer.history['recall'], 'b-', label='Recall', linewidth=2)
axes[1, 0].plot(epochs, trainer.history['f1'], 'r-', label='F1', linewidth=2)
axes[1, 0].set_xlabel('Epochs')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Metrics')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs, trainer.history['learning_rates'], 'purple', linewidth=2)
axes[1, 1].set_xlabel('Epochs')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print("\n" + "="*60)
print("📊 FINAL METRICS")
print("="*60)
print(f"Train Accuracy:      {trainer.history['train_acc'][-1]*100:.2f}%")
print(f"Validation Accuracy: {trainer.history['val_acc'][-1]*100:.2f}%")
print(f"Precision:           {trainer.history['precision'][-1]:.4f}")
print(f"Recall:              {trainer.history['recall'][-1]:.4f}")
print(f"F1 Score:            {trainer.history['f1'][-1]:.4f}")
print("="*60 + "\n")

## 🎯 Step 8: Evaluate on Test Set

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

# Load best model
checkpoint = torch.load(output_dir / 'best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Evaluate on test set
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Calculate metrics
test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
test_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average='macro')
cm = confusion_matrix(all_labels, all_preds)

print("\n" + "="*60)
print("🎯 TEST SET RESULTS")
print("="*60)
print(f"Accuracy:  {test_acc*100:.2f}%")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")
print("="*60 + "\n")

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 💾 Step 9: Verify Saved Files

In [ ]:
print(f"\n📁 Files saved in: {config['output_dir']}\n")
print("="*60)

for file in output_dir.glob('*'):
    if file.is_file():
        size = file.stat().st_size / 1024 / 1024  # MB
        print(f"✅ {file.name:40s} ({size:.2f} MB)")

print("="*60)
print(f"\n💡 These files are saved in Google Drive and will persist after Colab disconnects!")
print(f"📂 Check your Drive at: {config['output_dir']}\n")

## 🎉 Training Complete!

### Your Results:
- ✅ **Model trained** with state-of-the-art techniques
- ✅ **Checkpoint saved** to Google Drive
- ✅ **Metrics logged** for analysis
- ✅ **Ready for inference** on new images

### Next Steps:
1. **Download model** from Google Drive if needed
2. **Use for inference** on new images
3. **Analyze results** - check confusion matrix for insights
4. **Improve further** - try different hyperparameters

### Tips:
- To retrain: Just re-run Cell 6
- To use different model: Change `model_name` in Cell 3
- To adjust accuracy: Modify `epochs` or `learning_rate`
- Results are automatically saved to Google Drive!

---

**Happy Training! 🚀**